# JTB 九州ふっこう応援割 クーポン監視（Colab版）

各県のクーポンページを1秒おきに見て、**変化があったときだけ**スマホに通知し、Googleドライブに記録します。PCからは音が出ません。

## 準備（最初の1回だけ）
1. スマホに **ntfy** アプリを入れる（iPhone: App Store / Android: Google Play で「ntfy」）
2. アプリの「＋」で、下の `NTFY_TOPIC` と**同じ名前**のトピックを購読する
   - トピック名を知っている人には通知が見えます。他人と被らない長めの名前にしてください
3. MyJTB にスマホでログインしておく（通知をタップ → そのまま取得・予約できるように）

## 実行
上から順にセルを実行します（▶）。最後のセルが動いている間ずっと監視が続きます。止めるときは ■ を押します。

## 注意
- **このタブは開いたままにしてください。** 無料版の Colab は、しばらく操作がないと切断され、長くても半日ほどで止まります。止まるとスマホに「停止しました」と通知が届きます（落ち方によっては届きません）
- Colab は Google のサーバーから接続するので、JTB 側にブロックされることがあります。そのときは「監視できていません」と通知が届きます。その場合は PC で `python jtb_watch.py --ntfy トピック名` を実行してください（PC からも音は出ません）

## 1. 設定

In [ ]:
NTFY_TOPIC = "jtb-kyushu-ここを自分だけの名前に変える"
ONLY = []          # 監視する県を絞る場合。例: ["熊本", "長崎"]（空なら全部）
INTERVAL = 1.0     # 取得間隔(秒)
SAVE_TO_DRIVE = True   # True: Googleドライブ「マイドライブ/jtb_watch」に記録を残す

## 2. 準備（ドライブ接続・プログラム作成）

In [ ]:
import os
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/jtb_watch"
else:
    OUT_DIR = "/content/jtb_watch"
os.makedirs(OUT_DIR, exist_ok=True)
ARGS = ["--ntfy", NTFY_TOPIC, "--out", OUT_DIR, "--interval", str(INTERVAL)] + (["--only", *ONLY] if ONLY else [])
print("記録先:", OUT_DIR)

In [ ]:
%%writefile jtb_watch.py
#!/usr/bin/env python3
"""JTB「九州ふっこう応援割」クーポンページ変化監視（Google Colab / PC 両対応）。

各県のクーポンページを1秒おきに取得し、内容が変わったときだけ
  - スマホへプッシュ通知（ntfy。音はスマホ側だけ、PCは無音）
  - 記録（events.csv に差分、snapshots/ に変化後のHTML）
を行う。変化がなければ何も記録しない。

判定は「特定の文言を探す」のではなく「ページの文字・ボタンが変わったか」で行う。
変化した行がどのクーポン区分（「【宿泊】泊数指定なし」など）かも記録する。
起動直後に数回取得して、毎回変わる部分（時刻・トークン等）を自動で無視対象にする。
数字だけの変化（時刻・閲覧数など）も無視する。

使い方:
    python jtb_watch.py --ntfy 好きなトピック名           # 監視開始
    python jtb_watch.py --ntfy 好きなトピック名 --test    # 通知テスト
    python jtb_watch.py --inspect                         # 取得できた文字・ボタンを表示（動作確認）
    python jtb_watch.py --only 熊本 長崎 --interval 2
Colab では jtb_watch_colab.ipynb を使う。
"""
import argparse
import csv
import difflib
import os
import re
import threading
import time
from datetime import datetime, timedelta, timezone

import requests
from bs4 import BeautifulSoup

JST = timezone(timedelta(hours=9))
PAGE = "https://www.jtb.co.jp/myjtb/campaign/coupon/detail/{}/page.asp"

TARGETS = {
    "熊本": PAGE.format("2609kyushukumamoto"),
    "福岡": PAGE.format("2609kyushufukuoka"),
    "佐賀": PAGE.format("2609kyushusaga"),
    "長崎": PAGE.format("2609kyushunagasaki"),
    "鹿児島": PAGE.format("2609kyushukagoshima"),
}

# 差分の追加行/削除行にこれらが含まれたら、通知タイトルに「復活?」「終了?」を付ける（目安）。
# ページ文言が分かったら調整する。判定に関係なく、変化があれば必ず通知・記録はする。
REVIVE_WORDS = ["取得する", "獲得する", "クーポンを取得", "予約する", "配布中"]
END_WORDS = ["終了", "上限", "予定枚数", "在庫", "受付停止", "なくなり"]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "ja,en;q=0.8",
}

WARMUP_FETCHES = 4        # 起動時に取得して揺れる行を学習する回数
ERROR_NOTIFY_AFTER = 10   # 連続失敗がこの回数に達したら異変として通知・記録
MAX_BACKOFF = 60          # 失敗時の待ち時間の上限(秒)
HEARTBEAT_SEC = 300       # 画面にだけ出す生存表示の間隔（記録はしない）


def now():
    return datetime.now(JST)


def log(msg):
    print(f"[{now():%m/%d %H:%M:%S}] {msg}", flush=True)


def extract_lines(html):
    """ページを上から順に「見える文字の行」に分解する。
    ボタン・リンクは押せる/押せない状態も1行にして、その位置に差し込む。
    同じ文言（「配布終了いたしました」が複数枚など）もまとめずに残す。"""
    soup = BeautifulSoup(html, "html.parser")
    for t in soup(["script", "style", "noscript", "template", "svg"]):
        t.decompose()
    for el in soup.find_all(["a", "button", "input"]):
        if el.name == "input" and el.get("type", "").lower() == "hidden":
            continue
        label = " ".join(el.get_text(" ", strip=True).split()) or el.get("value", "") or el.get("alt", "")
        flags = []
        if el.has_attr("disabled"):
            flags.append("disabled")
        cls = " ".join(el.get("class", []))
        if any(k in cls for k in ("disable", "inactive", "end", "off", "soldout")):
            flags.append(f"class={cls}")
        if not label and not flags:
            continue
        href = (el.get("href") or "").split("?")[0]
        el.insert_before(f"\n[{el.name}] {label} {href} {' '.join(flags)}".rstrip() + "\n")
    return [" ".join(raw.split()) for raw in soup.get_text("\n").splitlines() if raw.strip()]


def section_of(lines, idx):
    """idx 行目が属するクーポン区分（直前の「【宿泊】泊数指定なし」のような「【」で始まる行）。"""
    for i in range(min(idx, len(lines) - 1), -1, -1):
        if lines[i].startswith("【"):
            return lines[i]
    return ""


def normalize(line):
    """比較用。トークン等の長いランダム文字列と数字を伏せる（時刻・アクセス数などで誤検知しないため）。"""
    line = re.sub(r"[A-Za-z0-9_\-]{20,}", "*", line)
    return re.sub(r"[0-9０-９]+", "#", line)


def hint(added, removed):
    """差分から「復活?」「終了?」の目安を付ける。押せないボタン(disabled)は終了側に数える。"""
    def has(words, s):
        return any(w in s for w in words)
    revive = sum(has(REVIVE_WORDS, s) and "disabled" not in s for s in added)
    revive += sum(has(END_WORDS, s) for s in removed)
    end = sum(has(REVIVE_WORDS, s) for s in removed)
    end += sum(has(END_WORDS, s) or (has(REVIVE_WORDS, s) and "disabled" in s) for s in added)
    if revive > end:
        return "復活?"
    if end > revive:
        return "終了?"
    return "変化"


class Notifier:
    def __init__(self, ntfy_topic=None, ntfy_server="https://ntfy.sh", discord_webhook=None):
        self.ntfy_topic = ntfy_topic
        self.ntfy_server = ntfy_server.rstrip("/")
        self.discord = discord_webhook

    def send(self, title, message, url=None, priority=4):
        log(f"通知: {title} / {message.splitlines()[0] if message else ''}")
        if self.ntfy_topic:
            body = {"topic": self.ntfy_topic, "title": title, "message": message[:3500],
                    "priority": priority, "tags": ["ticket"]}
            if url:
                body["click"] = url
                body["actions"] = [{"action": "view", "label": "ページを開く", "url": url}]
            self._post(self.ntfy_server, json=body)
        if self.discord:
            self._post(self.discord, json={"content": f"**{title}**\n{message[:1800]}\n{url or ''}"})

    def _post(self, url, **kw):
        try:
            requests.post(url, timeout=10, **kw).raise_for_status()
        except Exception as e:  # 通知失敗でも監視は止めない
            log(f"通知失敗: {e}")


class Recorder:
    FIELDS = ["時刻", "県", "クーポン区分", "種別", "判定", "追加された行", "消えた行", "URL", "スナップショット"]

    def __init__(self, out_dir):
        self.out_dir = out_dir
        self.snap_dir = os.path.join(out_dir, "snapshots")
        os.makedirs(self.snap_dir, exist_ok=True)
        self.csv_path = os.path.join(out_dir, "events.csv")
        self.lock = threading.Lock()

    def write(self, pref, kind, judge, url, added=(), removed=(), html=None, where=""):
        t = now()
        snap = ""
        if html is not None:
            snap = os.path.join(self.snap_dir, f"{t:%Y%m%d_%H%M%S}_{pref}.html")
            with open(snap, "w", encoding="utf-8") as f:
                f.write(html)
        with self.lock:
            new = not os.path.exists(self.csv_path)
            # utf-8-sig: Excel で開いても文字化けしない
            with open(self.csv_path, "a", newline="", encoding="utf-8-sig") as f:
                w = csv.writer(f)
                if new:
                    w.writerow(self.FIELDS)
                w.writerow([f"{t:%Y-%m-%d %H:%M:%S}", pref, where, kind, judge,
                            "\n".join(added), "\n".join(removed), url, snap])


class Watcher:
    def __init__(self, pref, url, interval, notifier, recorder):
        self.pref, self.url, self.interval = pref, url, interval
        self.notifier, self.recorder = notifier, recorder
        self.session = requests.Session()
        self.session.headers.update(HEADERS)
        self.ignore = set()
        self.items = None  # [(比較用の行, 元の行), ...]
        self.html = None
        self.errors = 0
        self.in_error = False
        self.checks = 0

    def fetch(self):
        """(html, [(比較用の行, 元の行), ...]) を返す。比較は数字を伏せた行、記録は元の行で行う。"""
        r = self.session.get(self.url, timeout=10)
        if r.status_code >= 400:
            raise RuntimeError(f"HTTP {r.status_code}")
        r.encoding = r.encoding if r.encoding and r.encoding.lower() != "iso-8859-1" else r.apparent_encoding
        raw = extract_lines(r.text)
        # ログイン画面などへ飛ばされたら、それも「変化」として拾う
        if r.url.split("?")[0] != self.url:
            raw.insert(0, f"[redirect] {r.url.split('?')[0]}")
        return r.text, [(normalize(x), x) for x in raw if normalize(x) not in self.ignore]

    def warmup(self):
        """数回取得して、毎回変わる行（時刻やトークン）を無視対象として覚える。"""
        samples = []
        for _ in range(WARMUP_FETCHES):
            html, items = self.fetch()
            samples.append({n for n, _ in items})
            time.sleep(self.interval)
        self.ignore = set.union(*samples) - set.intersection(*samples)
        self.html, self.items = html, [(n, x) for n, x in items if n not in self.ignore]
        log(f"{self.pref}: 監視開始（{len(self.items)}行を監視 / 揺れる{len(self.ignore)}行は無視）")

    def check(self):
        html, items = self.fetch()
        old = [n for n, _ in self.items]
        new = [n for n, _ in items]
        if old == new:
            return
        old_raw = [x for _, x in self.items]
        new_raw = [x for _, x in items]
        added, removed, sections = [], [], []
        for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, old, new, autojunk=False).get_opcodes():
            if op == "equal":
                continue
            removed += old_raw[i1:i2]
            added += new_raw[j1:j2]
            if i2 > i1:
                sections.append(section_of(old_raw, i2 - 1))
            if j2 > j1:
                sections.append(section_of(new_raw, j2 - 1))
        sections = list(dict.fromkeys(x for x in sections if x)) or ["（区分不明）"]
        judge = hint(added, removed)
        where = " / ".join(sections)
        self.recorder.write(self.pref, "ページ変化", judge, self.url, added, removed, html, where)
        msg = where + "\n" + "\n".join([f"＋ {x}" for x in added[:8]] + [f"－ {x}" for x in removed[:8]])
        self.notifier.send(f"【{judge}】JTB {self.pref}", msg, self.url,
                           priority=5 if judge == "復活?" else 4)
        self.html, self.items = html, items

    def run(self, stop):
        backoff = self.interval
        while not stop.is_set():
            if self.items is None:
                try:
                    self.warmup()
                except Exception as e:
                    self._error(e)
                    stop.wait(min(MAX_BACKOFF, backoff))
                    backoff *= 2
                    continue
            started = time.monotonic()
            try:
                self.check()
                self.checks += 1
                if self.in_error:
                    self.recorder.write(self.pref, "取得復旧", "", self.url)
                    self.notifier.send(f"JTB {self.pref} 監視復旧", "再び取得できています", self.url, 3)
                self.errors, self.in_error, backoff = 0, False, self.interval
            except Exception as e:
                self._error(e)
                backoff = min(MAX_BACKOFF, backoff * 2)
                stop.wait(backoff)
                continue
            stop.wait(max(0.0, self.interval - (time.monotonic() - started)))

    def _error(self, e):
        self.errors += 1
        if self.errors <= 3 or self.errors % 20 == 0:
            log(f"{self.pref}: 取得失敗({self.errors}) {e}")
        if self.errors == ERROR_NOTIFY_AFTER and not self.in_error:
            self.in_error = True
            self.recorder.write(self.pref, "取得失敗", str(e), self.url)
            self.notifier.send(f"JTB {self.pref} 監視できていません",
                               f"{self.errors}回連続で取得失敗: {e}\nブロックされている可能性があります",
                               self.url, 4)


def parse_args(argv=None):
    ap = argparse.ArgumentParser(description="JTBクーポンページ変化監視")
    ap.add_argument("--ntfy", help="ntfy のトピック名（スマホアプリで同じ名前を購読）")
    ap.add_argument("--ntfy-server", default="https://ntfy.sh")
    ap.add_argument("--discord", help="Discord Webhook URL（任意）")
    ap.add_argument("--out", default="jtb_log", help="記録の保存先フォルダ")
    ap.add_argument("--interval", type=float, default=1.0, help="各ページの取得間隔(秒)")
    ap.add_argument("--only", nargs="+", help="監視する県を絞る（例: --only 熊本 長崎）")
    ap.add_argument("--test", action="store_true", help="通知テストして終了")
    ap.add_argument("--inspect", action="store_true", help="取得結果を表示して終了")
    return ap.parse_args(argv)


def main(argv=None):
    args = parse_args(argv)
    targets = {k: v for k, v in TARGETS.items() if not args.only or k in args.only}
    notifier = Notifier(args.ntfy, args.ntfy_server, args.discord)

    if args.test:
        notifier.send("JTB監視テスト", "この通知が届けば設定OKです", next(iter(TARGETS.values())), 4)
        return

    if args.inspect:
        for pref, url in targets.items():
            w = Watcher(pref, url, args.interval, notifier, None)
            try:
                _, items = w.fetch()
                lines = [x for _, x in items]
            except Exception as e:
                print(f"\n===== {pref}: 取得失敗 {e}")
                continue
            print(f"\n===== {pref} ({len(lines)}行) {url}")
            for s in lines:
                print("  " + s)
        return

    if not args.ntfy and not args.discord:
        log("注意: 通知先が未設定です（--ntfy か --discord）。画面表示と記録のみ行います")

    recorder = Recorder(args.out)
    stop = threading.Event()
    watchers = [Watcher(p, u, args.interval, notifier, recorder) for p, u in targets.items()]
    threads = []
    for i, w in enumerate(watchers):
        th = threading.Thread(target=w.run, args=(stop,), daemon=True)
        th.start()
        threads.append(th)
        time.sleep(args.interval / max(1, len(watchers)))  # 同時アクセスを避けてずらす
    log(f"{len(watchers)}ページを{args.interval:g}秒おきに監視中 / 記録先: {os.path.abspath(args.out)}")
    notifier.send("JTB監視を開始しました", "、".join(targets), priority=2)

    try:
        while True:
            time.sleep(HEARTBEAT_SEC)
            log("稼働中: " + " / ".join(
                f"{w.pref} {'失敗中' if w.in_error else ('準備中' if w.items is None else f'{w.checks}回')}"
                for w in watchers))
    except KeyboardInterrupt:
        pass
    finally:
        stop.set()
        log("監視を停止しました")
        notifier.send("JTB監視が停止しました", "再開するにはセルを実行し直してください", priority=3)


if __name__ == "__main__":
    main()


## 3. 通知テスト（スマホに届くか確認）

In [ ]:
import importlib, jtb_watch
importlib.reload(jtb_watch)
jtb_watch.main(ARGS + ["--test"])

## 4. 動作確認（任意）
JTBのページが取れているか、クーポンの状態を表す文字・ボタンが含まれているかを確認します。`HTTP 403` などで失敗する場合は、Colab からの接続がブロックされています。

In [ ]:
jtb_watch.main(ARGS + ["--inspect"])

## 5. 監視開始
止めるまで動き続けます。変化があったときだけ、スマホへの通知と `events.csv`・`snapshots/` への記録を行います。

In [ ]:
jtb_watch.main(ARGS)